[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/kepler/kepler.ipynb)

# Spinor Regularization of Kepler Orbits

This example computes close passages of eccentric Kepler orbits by spinor regularization. A highly eccentric orbit swings past the attracting centre so quickly that fixed steps in physical time cut across the turn. Advancing a regularized clock in proportion to the distance from the centre slows the step near periapsis and turns the singular inverse-square attraction into a smooth harmonic oscillator.

The body's position is modelled as `spinor >> x.dual()`, which turns and scales the unit reference point by an even spinor whose reverse norm squared `spinor.scalar_norm_squared()` equals the orbital radius. Differentiating the bilinear sandwich map `Spinor >> x.dual()` in both factors cancels the extra rotor freedom in the plane perpendicular to `x.dual()`, while `velocity * spinor * ~x.dual() / 2` lifts the initial velocity without exciting that gauge spin. Because the regularized clock advances by `spinor.scalar_norm_squared()`, the spinor satisfies a linear oscillator equation with constant coefficient proportional to the orbital energy, following [Hestenes and Lounesto (1983)](https://davidhestenes.net/misc/Hestenes_1983_-_Geometry_of_Spinor_Regularization.pdf).

In Kustaanheimo–Stiefel matrix notation, a four-component column $\mathbf u = (u_1, u_2, u_3, u_4)^\mathsf{T} \in \mathbb R^4$ maps to Cartesian coordinates with a padded fourth zero $(\mathbf x, 0)^\mathsf{T} = L(\mathbf u)\,\mathbf u$ through the component-shuffled $4 \times 4$ matrix $L(\mathbf u)$ whose rows permute $(u_1, -u_2, -u_3, u_4)$, $(u_2, u_1, -u_4, -u_3)$, $(u_3, u_4, u_1, u_2)$, and $(u_4, -u_3, u_2, -u_1)$. Under $dt/ds = \mathbf u^\mathsf{T}\mathbf u$, the oscillator equation $d^2\mathbf u/ds^2 = \tfrac12 E\,\mathbf u$ must be paired with the explicit bilinear gauge constraint $u_1 u_4' - u_2 u_3' + u_3 u_2' - u_4 u_1' = 0$ on the initial derivative $\mathbf u' = d\mathbf u/ds$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterable, Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D as ga
from examples.util.animation import save_figure, show_states
from examples.mechanics.kepler import render

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# Positions and velocities are antivectors with intrinsic orientation.
Point = ga.gatype.antivector()
Spinor = ga.gatype.even()

# Unit point along x carried by a spinor in both factors of the sandwich product.
placement = Spinor >> mv.x.dual()                                        # [] Point- <- (Spinor+, Spinor+)
# Gauge-free velocity lift with zero spin in the plane perpendicular to the reference point.
velocity_lift = Point * Spinor * ~mv.x.dual() / 2                        # [] Spinor+ <- (Point-, Spinor+)

## Lifting an orbit to a spinor

Each orbit starts at apoapsis with a scaled rotor as its initial spinor and an initial spinor rate lifted from the velocity without gauge spin. In the regularized clock, the spinor follows a closed-form harmonic oscillation at frequency `(-energy / 2).square_root()`, and one full physical orbit corresponds to half a spinor cycle because `(-spinor) >> x.dual()` equals `spinor >> x.dual()`. Because the spinor turns strictly within the orbital plane, the same algebraic lift applies in any spatial dimension.

In Kustaanheimo–Stiefel matrix notation, representing the map by a square matrix $L(\mathbf u)$ linear in $\mathbf u$ with $L(\mathbf u)L(\mathbf u)^\mathsf{T} = (\mathbf u^\mathsf{T}\mathbf u) I$ restricts the construction by Hurwitz's sum-of-squares theorem to dimensions $1$, $2$, $4$, and $8$.

In [ ]:
gravity = 1.0
semimajor_axis = 1.0
eccentricities = np.array([0.6, 0.9, 0.97])
n_cases = len(eccentricities)
featured_case = 2
n_lift_samples = 64

In [ ]:
def oscillator_spinor(spinor: Spinor, rate: Spinor, frequency: Scalar, parameter: Scalar) -> Spinor:
    """Harmonic-oscillator evolution advances `spinor` (`[...] Spinor+`) at angular `frequency` in the regularized clock `parameter`."""
    phase = frequency * parameter                                        # [...] Scalar+
    return spinor * phase.cos() + rate * (phase.sin() / frequency)


# Start at apoapsis, where spinor.scalar_norm_squared() gives the initial radius.
apoapsis_radius = semimajor_axis * (1 + eccentricities)                  # [n_cases]
apoapsis_speed = np.sqrt(gravity / semimajor_axis * (1 - eccentricities) / (1 + eccentricities))  # [n_cases]
spinor = mv.rotor() * np.sqrt(apoapsis_radius)                           # [n_cases] Spinor+
apoapsis_velocity = mv.y.dual() * apoapsis_speed                         # [n_cases] Point-
energy = apoapsis_velocity.scalar_norm_squared() / 2 - gravity / spinor.scalar_norm_squared()  # [n_cases] Scalar+
rate = velocity_lift(apoapsis_velocity, spinor)                          # [n_cases] Spinor+
frequency = (-energy / 2).square_root()                                  # [n_cases] Scalar+

# Equal regularized steps resolve periapsis over half a spinor cycle.
lift_parameter = np.linspace(0, np.pi, n_lift_samples + 1)[:, None] / frequency  # [n_lift_samples + 1, n_cases] Scalar+
lift_spinors = oscillator_spinor(spinor, rate, frequency, lift_parameter)  # [n_lift_samples + 1, n_cases] Spinor+
lift_positions = lift_spinors >> mv.x.dual()                             # [n_lift_samples + 1, n_cases] Point-
save_figure(render.lift_figure(lift_positions[:, featured_case], lift_spinors[:, featured_case]), "kepler_spinor_lift");

The left panel shows the eccentric orbit (`0.97`) in the `xy` plane around the gold attracting centre, while the right panel plots the scalar and `xy` components of its spinor over one physical revolution. Equal steps in the regularized clock cluster automatically in physical space near periapsis, where the small orbital radius slows physical time relative to the spinor phase.

## Integrating in both clocks

Kick-drift-kick Verlet advances both formulations over one orbital period with the same step count, stepping position in physical time under the inverse-square acceleration and stepping the spinor in the regularized clock as a linear oscillator while accumulating physical time by Simpson's rule. Integrating the squared spinor norm in closed form gives the exact physical clock alongside the exact orbit for reference.

In [ ]:
n_steps = 128
n_reference_samples = 1024
period = 2 * np.pi * np.sqrt(semimajor_axis ** 3 / gravity)

In [ ]:
def physical_state(spinor: Spinor, rate: Spinor) -> tuple[Point, Point]:
    """The sandwich product and its derivative map `spinor` and `rate` to physical position and velocity (`[...] Point-`, `[...] Point-`)."""
    # The spinor times its reverse equals the distance from the attracting centre at the origin.
    radius = spinor.scalar_norm_squared()                                # [...] Scalar+
    # Differentiating both factors of the sandwich cancels the scalar gauge rate.
    velocity = (placement(rate, spinor) + placement(spinor, rate)) / radius  # [...] Point-
    return spinor >> mv.x.dual(), velocity


def exact_orbit(spinor: Spinor, rate: Spinor, frequency: Scalar,
                parameter: Scalar) -> tuple[Scalar, Point, Point]:
    """Closed-form oscillator motion and integrated radius give physical `(time, position, velocity)` at regularized clock `parameter`."""
    phase = frequency * parameter                                        # [...] Scalar+
    sine = phase.sin()                                                   # [...] Scalar+
    current_spinor = oscillator_spinor(spinor, rate, frequency, parameter)  # [...] Spinor+
    current_rate = rate * phase.cos() - spinor * (frequency * sine)      # [...] Spinor+
    # Physical time advances by the radius, slowing the regularized clock smoothly near the centre.
    initial_radius = spinor.scalar_norm_squared()                        # [...] Scalar+
    rate_radius = rate.scalar_norm_squared() / frequency.squared()       # [...] Scalar+
    overlap = spinor.scalar_product(~rate) / frequency                   # [...] Scalar+
    time = ((initial_radius + rate_radius) * parameter / 2
            + (initial_radius - rate_radius) * (2 * phase).sin() / (4 * frequency)
            + overlap * sine.squared() / frequency)                      # [...] Scalar+
    position, velocity = physical_state(current_spinor, current_rate)    # [...] Point-, Point-
    return time, position, velocity


initial_time = 0 * spinor.scalar_norm_squared()                          # [n_cases] Scalar+
initial_positions, initial_velocities = physical_state(spinor, rate)     # [n_cases] Point-, Point-
reference_parameter = np.linspace(0, np.pi, n_reference_samples + 1)[:, None] / frequency  # [n_reference_samples + 1, n_cases] Scalar+
reference_times, reference_positions, reference_velocities = exact_orbit(
    spinor, rate, frequency, reference_parameter)                        # [n_reference_samples + 1, n_cases] Scalar+, Point-, Point-

In [ ]:
def physical_verlet(position: Point, velocity: Point, time: Scalar,
                    gravity: float, step: float, n_steps: int) -> Iterator[tuple[Scalar, Point, Point]]:
    """Yields `(time, position, velocity)` states across `n_steps` equal kick-drift-kick steps in physical time."""
    acceleration = -gravity * position / position.norm() ** 3            # [...] Point-
    yield time, position, velocity
    for _ in range(n_steps):
        half_velocity = velocity + acceleration * (step / 2)             # [...] Point-
        position = position + half_velocity * step                       # [...] Point-
        acceleration = -gravity * position / position.norm() ** 3        # [...] Point-
        velocity = half_velocity + acceleration * (step / 2)             # [...] Point-
        time = time + step                                               # [...] Scalar+
        yield time, position, velocity


def regularized_verlet(spinor: Spinor, rate: Spinor, energy: Scalar, time: Scalar,
                       step: Scalar, n_steps: int) -> Iterator[tuple[Scalar, Point, Point]]:
    """Yields `(time, position, velocity)` states across `n_steps` kick-drift-kick oscillator steps in the regularized clock."""
    initial_position, initial_velocity = physical_state(spinor, rate)    # [...] Point-, Point-
    yield time, initial_position, initial_velocity
    for _ in range(n_steps):
        half_rate = rate + energy * spinor * (step / 4)                  # [...] Spinor+
        next_spinor = spinor + half_rate * step                          # [...] Spinor+
        # Simpson's rule integrates the quadratic drift radius over the regularized step.
        time = time + step / 6 * (spinor.scalar_norm_squared()
                                  + (spinor + next_spinor).scalar_norm_squared()
                                  + next_spinor.scalar_norm_squared())   # [...] Scalar+
        rate = half_rate + energy * next_spinor * (step / 4)             # [...] Spinor+
        spinor = next_spinor                                             # [...] Spinor+
        next_position, next_velocity = physical_state(spinor, rate)      # [...] Point-, Point-
        yield time, next_position, next_velocity


physical_samples = list(physical_verlet(
    initial_positions, initial_velocities, initial_time, gravity, period / n_steps, n_steps))
regularized_samples = list(regularized_verlet(
    spinor, rate, energy, initial_time, np.pi / frequency / n_steps, n_steps))
physical_positions = stack([sample[1] for sample in physical_samples])        # [n_steps + 1, n_cases] Point-
physical_velocities = stack([sample[2] for sample in physical_samples])       # [n_steps + 1, n_cases] Point-
regularized_positions = stack([sample[1] for sample in regularized_samples])  # [n_steps + 1, n_cases] Point-
regularized_velocities = stack([sample[2] for sample in regularized_samples]) # [n_steps + 1, n_cases] Point-
save_figure(render.passage_figure(reference_positions, physical_positions, regularized_positions), "kepler_passage");

Across the three columns for eccentricities `0.6`, `0.9`, and `0.97`, the upper row shows the full orbit and the lower row magnifies the boxed periapsis passage. The blue spinor-clock trajectory tracks the grey exact ellipse closely through periapsis, whereas the orange physical-clock trajectory with the same fixed step count cuts inside the turn and drifts off the ellipse at high eccentricity. Both Verlet integrators use uniform steps in their own clocks rather than adaptive step-size control.

## Convergence with the number of steps

Inverting the monotone physical clock of the analytic solution by bisection evaluates the exact position at each integrator's own sample times. Weighting the squared position discrepancy by elapsed physical time yields a root-mean-square trajectory error that treats clustered periapsis steps and wide apoapsis steps on equal physical footing.

In [ ]:
step_counts = np.array([64, 128, 256, 512])
n_step_counts = len(step_counts)
n_clock_iterations = 44

In [ ]:
def state_at_time(spinor: Spinor, rate: Spinor, frequency: Scalar,
                  time: Scalar, n_iterations: int) -> tuple[Scalar, Point, Point]:
    """Batched bisection over `n_iterations` steps inverts the monotone physical clock to evaluate `(time, position, velocity)` at `time`."""
    mean_radius = (spinor.scalar_norm_squared()
                   + rate.scalar_norm_squared() / frequency.squared()) / 2  # [...] Scalar+
    midpoint = time / mean_radius                                        # [...] Scalar+
    half_period = np.pi / frequency                                      # [...] Scalar+
    lower_bound, upper_bound = midpoint - half_period, midpoint + half_period  # [...] Scalar+, Scalar+
    for _ in range(n_iterations):
        midpoint = (lower_bound + upper_bound) / 2                       # [...] Scalar+
        earlier_mask = exact_orbit(spinor, rate, frequency, midpoint)[0] < time
        lower_bound = earlier_mask * midpoint + (1 - earlier_mask) * lower_bound  # [...] Scalar+
        upper_bound = earlier_mask * upper_bound + (1 - earlier_mask) * midpoint  # [...] Scalar+
    return exact_orbit(spinor, rate, frequency, (lower_bound + upper_bound) / 2)


def position_error(spinor: Spinor, rate: Spinor, frequency: Scalar,
                   states: Iterable[tuple[Scalar, Point, Point]], n_iterations: int) -> Scalar:
    """Trapezoidal quadrature over elapsed physical time computes the root-mean-square position error (`[...] Scalar+`) against the exact orbit."""
    sample_list = list(states)
    n_samples = len(sample_list)
    times = stack([sample[0] for sample in sample_list])                 # [n_samples, ...] Scalar+
    positions = stack([sample[1] for sample in sample_list])             # [n_samples, ...] Point-
    exact_positions = state_at_time(spinor, rate, frequency, times, n_iterations)[1]  # [n_samples, ...] Point-
    squared_errors = (positions - exact_positions).scalar_norm_squared()  # [n_samples, ...] Scalar+
    intervals = times[1:] - times[:-1]                                   # [n_samples - 1, ...] Scalar+
    # Weight by elapsed physical time so clustered periapsis samples do not dominate the norm.
    integral = ((squared_errors[1:] + squared_errors[:-1]) * intervals / 2).sum(axis=0)  # [...] Scalar+
    return (integral / (times[-1] - times[0])).square_root()


physical_errors = stack([
    position_error(
        spinor, rate, frequency,
        physical_verlet(initial_positions, initial_velocities, initial_time,
                        gravity, period / int(n_orbit_steps), int(n_orbit_steps)),
        n_clock_iterations)
    for n_orbit_steps in step_counts
]) / semimajor_axis                                                      # [n_step_counts, n_cases] Scalar+
regularized_errors = stack([
    position_error(
        spinor, rate, frequency,
        regularized_verlet(spinor, rate, energy, initial_time,
                           np.pi / frequency / int(n_orbit_steps), int(n_orbit_steps)),
        n_clock_iterations)
    for n_orbit_steps in step_counts
]) / semimajor_axis                                                      # [n_step_counts, n_cases] Scalar+
save_figure(render.convergence_figure(step_counts, physical_errors, regularized_errors), "kepler_convergence");

Once the close passage is resolved, both second-order Verlet integrators fall along parallel slope-`-2` lines on the log-log axes, while rising eccentricity shifts the orange physical-clock error upward by orders of magnitude relative to the blue spinor-clock curve. Both integrators preserve the orbital angular momentum `position & velocity`, whereas the regularized integrator's physical energy oscillates transiently during periapsis passage and returns over a full period.

## The spinor over two revolutions

Stepping uniformly in the regularized clock advances the spinor around one full oscillator cycle while the physical body completes two orbital revolutions, because opposite spinors `spinor` and `-spinor` sandwich `x.dual()` to the same position.

In [ ]:
n_frames = 180
frame_duration_ms = 40
n_revolutions = 2

In [ ]:
def lift_states(spinor: Spinor, rate: Spinor, frequency: Scalar,
                n_revolutions: int, n_frames: int) -> Iterator[tuple[Point, Spinor]]:
    """Yields `(position, spinor)` pairs (`[...] Point-`, `[...] Spinor+`) at `n_frames + 1` uniform regularized-clock samples over `n_revolutions` orbits."""
    for frame_index in range(n_frames + 1):
        parameter = frame_index * n_revolutions * np.pi / (n_frames * frequency)  # [...] Scalar+
        current_spinor = oscillator_spinor(spinor, rate, frequency, parameter)  # [...] Spinor+
        yield current_spinor >> mv.x.dual(), current_spinor


revolving_states = show_states(
    lift_states(spinor[featured_case], rate[featured_case], frequency[featured_case], n_revolutions, n_frames),
    render.lift_frames,
    name="kepler_spinor_lift",
    duration_ms=frame_duration_ms,
    n_frames=n_frames + 1,
);

As the magenta marker completes its first revolution around the gold attracting centre on the left, the spinor marker on the right travels halfway around its ellipse from the positive scalar axis to the negative scalar axis, returning to its starting spinor only at the end of the second physical orbit.

In [ ]:
# checks
assert ((placement(rate, spinor) - placement(spinor, rate)).norm() < 1e-10).all()
assert ((lift_positions[-1] - lift_positions[0]).norm() < 1e-10).all()
assert ((lift_spinors[-1] + spinor).norm() < 1e-10).all()
assert ((initial_velocities - apoapsis_velocity).norm() < 1e-10).all()
assert ((reference_times[-1] - period).abs() < 1e-10).all()
assert ((reference_positions[-1] - initial_positions).norm() < 1e-10).all()
assert (((reference_positions & reference_velocities) - (initial_positions & initial_velocities)).norm() < 1e-10).all()
assert (((physical_positions[-1] & physical_velocities[-1]) - (initial_positions & initial_velocities)).norm() < 1e-10).all()
assert (((regularized_positions[-1] & regularized_velocities[-1]) - (initial_positions & initial_velocities)).norm() < 1e-10).all()
assert (regularized_errors[-1] < regularized_errors[-2] / 3.5).all()
assert regularized_errors[-1, featured_case] < physical_errors[-1, featured_case] / 100
assert (revolving_states[n_frames // 2][0] - revolving_states[0][0]).norm() < 1e-10
assert (revolving_states[n_frames // 2][1] + revolving_states[0][1]).norm() < 1e-10
assert (revolving_states[-1][1] - revolving_states[0][1]).norm() < 1e-10